<!-- NB_DOC_INTRO_V1 -->
# Pont DataFrame ↔ Bases de donnees (Postgres / SQLite / Mongo)

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Templates pour **lire et ecrire des DataFrames** depuis/vers des bases de donnees usuelles : SQLite (built-in Python), PostgreSQL (via SQLAlchemy + psycopg2), MongoDB (via pymongo). Le notebook execute des **demos SQLite** end-to-end (no install required), et fournit les patterns Postgres / Mongo en code pret a executer.

Pour DuckDB (alternative analytique embarquee) : voir [BDD_DuckDB.ipynb](./BDD_DuckDB.ipynb). Pour ORM : [FastAPI_API.ipynb](./FastAPI_API.ipynb).

## Plan

1. SQLite execute (built-in)
2. PostgreSQL via SQLAlchemy + psycopg2 (patterns)
3. MongoDB via pymongo (patterns)
4. Bulk insert vs append (perf)
5. Gestion des secrets (env vars)
6. Pieges et anti-patterns
7. References


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import tempfile, os

SEED = 42
np.random.seed(SEED)

# === Dataset jouet ===
df = pd.DataFrame({
    "id": range(1, 11),
    "name": [f"user_{i}" for i in range(1, 11)],
    "score": np.random.uniform(0, 100, 10).round(2),
    "active": np.random.choice([True, False], 10),
})
print(df.head())

## 1. SQLite — execute end-to-end

SQLite est **built-in** Python (`sqlite3`). DB en 1 fichier, parfait pour POC / dev / tests.

In [ ]:
# === Creer une DB temporaire ===
db_path = os.path.join(tempfile.gettempdir(), "demo.sqlite")
if os.path.exists(db_path):
    os.remove(db_path)

con = sqlite3.connect(db_path)
print(f"SQLite DB : {db_path}")

# === DataFrame -> SQL ===
df.to_sql("users", con, if_exists="replace", index=False)
print(f"Inserted {len(df)} rows")

# === SQL -> DataFrame ===
df_back = pd.read_sql_query("SELECT * FROM users WHERE score > 50 ORDER BY score DESC", con)
print(f"\nQuery result :")
print(df_back)

In [ ]:
# === Requete avec parametres (anti-injection) ===
min_score = 70
df_filtered = pd.read_sql_query(
    "SELECT name, score FROM users WHERE score > ?",
    con, params=(min_score,)
)
print(df_filtered)

In [ ]:
# === Aggregations SQL ===
agg = pd.read_sql_query('''
    SELECT
        active,
        COUNT(*) AS n,
        AVG(score) AS mean_score,
        MAX(score) AS max_score
    FROM users
    GROUP BY active
''', con)
print(agg)

con.close()
os.remove(db_path)
print("\nDB cleanup OK")

## 2. PostgreSQL — via SQLAlchemy + psycopg2

```bash
pip install sqlalchemy psycopg2-binary python-dotenv
```

### Connexion
```python
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

# JAMAIS hardcoder les credentials !
DB_URL = f"""postgresql+psycopg2://{os.environ['PG_USER']}:{os.environ['PG_PASSWORD']}@{os.environ['PG_HOST']}:5432/{os.environ['PG_DB']}"""

engine = create_engine(DB_URL, pool_size=5, pool_pre_ping=True)
```

### DataFrame → table (insertion)
```python
df.to_sql("users", engine, if_exists="append", index=False, chunksize=10_000)
```

### Table → DataFrame (lecture)
```python
df_read = pd.read_sql("SELECT * FROM users WHERE active = TRUE", engine)
# Ou avec params :
df_read = pd.read_sql(
    text("SELECT * FROM users WHERE score > :s"),
    engine, params={"s": 70},
)
```

### Bulk insert ultra-rapide via COPY FROM
```python
from io import StringIO

def bulk_insert(engine, df, table_name):
    """~ 10-100× plus rapide que to_sql sur gros volumes (millions de lignes)."""
    with engine.raw_connection().cursor() as cur:
        buf = StringIO()
        df.to_csv(buf, index=False, header=False, sep="\t", na_rep="\\N")
        buf.seek(0)
        cur.copy_from(buf, table_name, sep="\t", null="\\N", columns=list(df.columns))
    engine.raw_connection().commit()
```


## 3. MongoDB — via pymongo

```bash
pip install pymongo
```

```python
from pymongo import MongoClient
import pandas as pd

client = MongoClient(os.environ["MONGO_URI"])
db = client["mydb"]
collection = db["users"]

# === DataFrame -> Mongo ===
collection.insert_many(df.to_dict("records"))

# === Mongo -> DataFrame ===
docs = list(collection.find({"score": {"$gt": 70}}, projection={"_id": 0}))
df_read = pd.DataFrame(docs)
print(df_read)

# === Aggregation pipeline ===
pipeline = [
    {"$match": {"active": True}},
    {"$group": {"_id": "$category", "mean_score": {"$avg": "$score"}, "n": {"$sum": 1}}},
    {"$sort": {"mean_score": -1}},
]
result = list(collection.aggregate(pipeline))
print(pd.DataFrame(result))
```


## 4. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Hardcoder credentials dans le code | `os.environ` + `.env` (gitignore !) |
| String concatenation pour SQL | `?` placeholders + params |
| `to_sql` sans `chunksize` pour gros volumes | `chunksize=10_000` ou `COPY FROM` |
| Pas de `pool_pre_ping=True` sur SQLAlchemy | Connexions stale = erreurs intermittentes |
| Connexion ouverte sans `with` | Pas garantie de fermeture |
| `read_sql("SELECT *")` sans LIMIT | OOM sur grosse table |
| `index=True` quand on ne veut pas l'index | Colonne "index" parasitee |
| Pas de TIMEOUT sur connexion | App hang silencieux |
| Mongo `find` sans projection | Telecharge tous les champs |

## 5. References

- **SQLAlchemy** : https://docs.sqlalchemy.org/
- **psycopg2** : https://www.psycopg.org/docs/
- **pymongo** : https://pymongo.readthedocs.io/
- **SQLite Python** : https://docs.python.org/3/library/sqlite3.html
- pandas IO SQL : https://pandas.pydata.org/docs/user_guide/io.html#sql-queries

### Voir aussi (collection)
- [BDD_DuckDB.ipynb](BDD_DuckDB.ipynb) — alternative columnar embarquee
- [FastAPI_API.ipynb](FastAPI_API.ipynb) — API + Postgres via SQLModel
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb) — manipulations pandas
